[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/41_capstone_projects/02_capstone_walkthrough.ipynb)

# Bài 02 — Capstone mẫu: chấm điểm tín dụng từ dữ liệu thô

**🎯 Mục tiêu**: đi trọn một dự án ML như ngoài thực tế — **dữ liệu thô → kiểm tra → làm sạch → EDA → chia theo thời gian → đặc trưng → baseline → mô hình → đánh giá → quyết định theo lợi nhuận → giải thích → kiểm tra theo nhóm → đóng gói → giám sát** — và thấy vì sao một mô hình "tốt trên validation" vẫn có thể gây lỗ khi thế giới thay đổi.

Mỗi phần ghi rõ chương liên quan. Đọc như một **báo cáo capstone mẫu**: mọi con số trong lời bình đều được tính ngay trong notebook.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/41_capstone_projects

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.optimize import brentq

from ml_course.datasets import CREDIT_MONTHLY_RATE, make_credit_applications

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
MARGIN_PER_YEAR = 0.05      # biên lợi nhuận ròng/năm trên dư nợ nếu khách trả đủ (sau chi phí vốn, vận hành)
LOSS_GIVEN_DEFAULT = 0.70   # tỉ lệ mất trên khoản vay nếu vỡ nợ (sau thu hồi)
ARTIFACTS = Path("artifacts")

## 1. Bài toán (design doc tóm tắt) — chương 39

| Mục | Nội dung |
|---|---|
| Quyết định | Duyệt / từ chối hồ sơ vay tiêu dùng, mỗi ngày, tự động cho hồ sơ phổ thông |
| Cách làm hiện tại (baseline nghiệp vụ) | Duyệt nếu **không trễ hạn** trong 12 tháng gần nhất **và** tỉ lệ nợ/thu nhập (DTI) < 45% |
| Kinh tế | Khách trả đủ: lãi ròng `5%/năm × khoản vay × kỳ hạn`. Vỡ nợ: mất `70% × khoản vay` |
| Metric kinh doanh | **Tổng lợi nhuận** trên hồ sơ của một giai đoạn chưa thấy |
| Metric mô hình | ROC-AUC (xếp hạng), **Brier / calibration** (vì quyết định dùng trực tiếp xác suất), PR-AUC |
| Guardrail | Tỉ lệ duyệt và hiệu năng theo vùng, nhóm tuổi; độ trễ < 50 ms |
| Nhãn | `defaulted` = quá hạn ≥ 90 ngày trong **12 tháng** đầu → nhãn trễ 12 tháng |

Quyết định tối ưu với xác suất vỡ nợ $p$, lãi $g$, lỗ $\ell$: duyệt khi lợi nhuận kỳ vọng $(1-p)\,g - p\,\ell > 0 \iff p < \dfrac{g}{g + \ell}$. Với $g = 0.05 \cdot \text{kỳ hạn}/12$ và $\ell = 0.7$ (theo tỉ lệ khoản vay), ngưỡng phụ thuộc kỳ hạn: 12 tháng → 6.7%; 24 tháng → 12.5%; 36 tháng → 17.6%. Ngưỡng tính trên **xác suất** → xác suất phải **được hiệu chỉnh** (chương 37), không chỉ xếp hạng đúng.

## 2. Dữ liệu thô — kiểm tra trước khi tin (chương 03, 04)

File `credit_applications_raw.csv` mô phỏng một bản xuất từ hệ thống nghiệp vụ.

In [ ]:
raw = pd.read_csv("../../data/sample/credit_applications_raw.csv")
print(raw.shape)
print(raw.head(4).T.to_string())

In [ ]:
def audit_raw(df: pd.DataFrame) -> pd.DataFrame:
    """Báo cáo vấn đề chất lượng dữ liệu — mỗi dòng một phát hiện."""
    findings = []

    def add(column, issue, count):
        if count:
            findings.append({"cột": column, "vấn đề": issue, "số dòng": int(count), "tỉ lệ": count / len(df)})

    add("(toàn bảng)", "dòng trùng hoàn toàn", df.duplicated().sum())
    add("application_id", "mã hồ sơ lặp", df.application_id.duplicated().sum())
    add("application_date", "định dạng dd/mm/yyyy (lẫn với yyyy-mm-dd)", df.application_date.astype(str).str.contains("/").sum())
    add("loan_amount_vnd", "kiểu chuỗi có dấu chấm phân cách nghìn", df.loan_amount_vnd.astype(str).str.contains(r"\.").sum())
    add("monthly_income_vnd", "thiếu", df.monthly_income_vnd.isna().sum())
    add("monthly_income_vnd", "< 1.000 (nghi nhập theo triệu đồng)", (df.monthly_income_vnd < 1000).sum())
    add("age", "ngoài [18, 100]", (~df.age.between(18, 100)).sum())
    add("employment_type", "thiếu", df.employment_type.isna().sum())
    add("employment_type", f"{df.employment_type.nunique()} cách viết cho 4 loại", (~df.employment_type.isin(["salaried", "self_employed", "contract", "unemployed"]) & df.employment_type.notna()).sum())
    add("region", f"{df.region.nunique()} cách viết cho 4 vùng", (~df.region.isin(["Hà Nội", "TP.HCM", "Đà Nẵng", "Khác"])).sum())
    add("defaulted", "nhãn chưa có (hồ sơ chưa đủ 12 tháng)", df.defaulted.isna().sum())
    add("early_dpd30_3m", "THÔNG TIN SAU QUYẾT ĐỊNH — cấm dùng làm đặc trưng", len(df))
    return pd.DataFrame(findings)


print(audit_raw(raw).round(3).to_string(index=False))
print("\ncác cách viết employment_type:", sorted(raw.employment_type.dropna().unique()))

Ba loại vấn đề cần xử lý khác nhau:
- **Lỗi định dạng** (ngày, số tiền dạng chuỗi, danh mục viết khác nhau) → sửa được chính xác bằng quy tắc.
- **Lỗi giá trị** (thu nhập theo triệu, tuổi 999) → sửa khi chắc chắn (thu nhập < 1.000 đồng là bất khả), còn lại đặt thành thiếu.
- **Vấn đề ngữ nghĩa** — quan trọng nhất và máy không tự biết: `defaulted` thiếu vì nhãn **chưa trưởng thành** (không phải lỗi); `early_dpd30_3m` là thông tin có **sau** khi duyệt vay — dùng làm đặc trưng là rò rỉ (phần 6).

## 3. Làm sạch có hợp đồng dữ liệu (chương 04, 40)

Hàm làm sạch là **code sản phẩm**: dùng lại khi phục vụ (phần 13), nên phải tất định và có kiểm tra đầu ra ("hợp đồng dữ liệu").

In [ ]:
EMPLOYMENT_TYPES = ["salaried", "self_employed", "contract", "unemployed", "unknown"]
REGION_MAP = {"tp.hcm": "TP.HCM", "ho chi minh": "TP.HCM", "hcm": "TP.HCM", "tp hcm": "TP.HCM", "hà nội": "Hà Nội", "ha noi": "Hà Nội",
              "hn": "Hà Nội", "đà nẵng": "Đà Nẵng", "da nang": "Đà Nẵng", "khác": "Khác"}


def clean_applications(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.drop_duplicates().drop_duplicates("application_id").copy()
    dates = df.application_date.astype(str)
    df["application_date"] = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce").fillna(
        pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce"))
    df["loan_amount_vnd"] = df.loan_amount_vnd.astype(str).str.replace(".", "", regex=False).astype(float)
    income = df.monthly_income_vnd.astype(float)
    df["monthly_income_vnd"] = np.where(income < 1000, income * 1e6, income)          # nhập theo triệu → đồng
    df["age"] = df.age.where(df.age.between(18, 100))                                  # tuổi vô lý → thiếu
    employment = df.employment_type.astype("string").str.strip().str.lower().str.replace("[- ]", "_", regex=True)
    df["employment_type"] = employment.replace({"hop_dong": "contract"}).fillna("unknown").astype(str)
    df["region"] = df.region.astype(str).str.strip().str.lower().map(REGION_MAP)
    return df.sort_values(["application_date", "application_id"]).reset_index(drop=True)


def validate_clean(df: pd.DataFrame) -> list[str]:
    """Hợp đồng dữ liệu cho bảng đã làm sạch; rỗng = đạt."""
    problems = []
    if df.application_id.duplicated().any():
        problems.append("trùng application_id")
    if df.application_date.isna().any():
        problems.append("ngày không đọc được")
    if not set(df.employment_type) <= set(EMPLOYMENT_TYPES):
        problems.append(f"loại việc làm lạ: {set(df.employment_type) - set(EMPLOYMENT_TYPES)}")
    if df.region.isna().any():
        problems.append("vùng không nhận diện được")
    if (df.monthly_income_vnd < 1e6).any() or (df.loan_amount_vnd <= 0).any():
        problems.append("số tiền ngoài miền hợp lệ")
    if df.age.dropna().between(18, 100).mean() < 1:
        problems.append("tuổi ngoài miền")
    return problems


apps = clean_applications(raw)
print(f"{len(raw):,} dòng thô → {len(apps):,} hồ sơ | hợp đồng dữ liệu: {validate_clean(apps) or 'ĐẠT'}")
print(apps.dtypes.to_string())

Vì dữ liệu là mô phỏng, ta có **bản gốc sạch** để kiểm chứng hàm làm sạch (thực tế: đối chiếu một mẫu ngẫu nhiên với hệ thống nguồn hoặc kiểm tra thủ công):

In [ ]:
truth = make_credit_applications(12000, seed=42).set_index("application_id").loc[apps.application_id]
known_income = apps.monthly_income_vnd.notna().values
known_age = apps.age.notna().values
checks = {
    "ngày": (apps.application_date.values == truth.application_date.values).mean(),
    "khoản vay": (apps.loan_amount_vnd.values == truth.loan_amount_vnd.values).mean(),
    "thu nhập (khi có)": np.isclose(apps.monthly_income_vnd.values[known_income], truth.monthly_income_vnd.values[known_income]).mean(),
    "tuổi (khi hợp lệ)": (apps.age.values[known_age] == truth.age.values[known_age]).mean(),
    "loại việc làm (khi có)": (apps.employment_type[apps.employment_type != "unknown"].values == truth.employment_type[(apps.employment_type != "unknown").values].values).mean(),
    "vùng": (apps.region.values == truth.region.values).mean(),
}
print({k: f"{v:.2%}" for k, v in checks.items()})

## 4. EDA hướng câu hỏi (chương 03)

EDA không phải vẽ mọi thứ — mà trả lời các câu hỏi của design doc: nhãn phân bố theo thời gian thế nào? Rủi ro phụ thuộc gì?

In [ ]:
apps["half"] = apps.application_date.dt.year.astype(str) + "-H" + np.where(apps.application_date.dt.month <= 6, "1", "2")
by_half = apps.groupby("half").agg(hồ_sơ=("application_id", "size"), có_nhãn=("defaulted", "count"),
                                   tỉ_lệ_vỡ_nợ=("defaulted", "mean"), nhãn_sớm=("early_dpd30_3m", "mean"))
print(by_half.round(3).to_string())

In [ ]:
annuity = CREDIT_MONTHLY_RATE / (1 - (1 + CREDIT_MONTHLY_RATE) ** -apps.term_months)
apps["dti"] = apps.loan_amount_vnd * annuity / apps.monthly_income_vnd
labeled = apps[apps.defaulted.notna() & (apps.application_date < "2024-01-01")]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
by_half.tỉ_lệ_vỡ_nợ.plot(kind="bar", ax=axes[0], color=["C0"] * 4 + ["C3"] + ["C7"] * 2)
axes[0].set_title("Tỉ lệ vỡ nợ theo nửa năm (xám: chưa có nhãn)")
labeled.groupby(pd.qcut(labeled.dti, 10), observed=True).defaulted.mean().plot(ax=axes[1], marker="o")
axes[1].set_title("Vỡ nợ theo thập phân vị DTI (2022–2023)")
axes[1].set_xticks([])
labeled.groupby("employment_type").defaulted.mean().sort_values().plot(kind="barh", ax=axes[2])
axes[2].set_title("Vỡ nợ theo loại việc làm (2022–2023)")
plt.tight_layout()
plt.show()

Ba điều EDA cho thấy:
- Tỉ lệ vỡ nợ ổn định ~7–8% trong 2022–2023 rồi **nhảy vọt** ở nửa đầu 2024 (cú sốc kinh tế); hồ sơ từ 07/2024 chưa có nhãn.
- Rủi ro tăng đều theo **DTI**; người thất nghiệp và lao động hợp đồng rủi ro cao hơn hẳn.
- `nhãn_sớm` (quá hạn 30 ngày trong 3 tháng đầu) **đi cùng** tỉ lệ vỡ nợ và có sớm hơn 9 tháng — ứng viên cho giám sát (phần 14).

## 5. Chia dữ liệu theo thời gian (chương 15, 39)

Kịch bản: ta đứng ở **07/2025** và đánh giá hồi cứu — mô hình học trên hồ sơ đến 06/2023, chọn trên 2023-H2, kiểm tra **một lần** trên 2024-H1. Chia ngẫu nhiên sẽ trộn tương lai vào quá khứ và che mất cú sốc.

In [ ]:
splits = {
    "train": (apps.application_date < "2023-07-01"),
    "validation": (apps.application_date >= "2023-07-01") & (apps.application_date < "2024-01-01"),
    "test": (apps.application_date >= "2024-01-01") & (apps.application_date < "2024-07-01"),
    "production (chưa nhãn)": apps.defaulted.isna(),
}
for name, mask in splits.items():
    print(f"{name:24s}: {mask.sum():5,} hồ sơ | {apps.application_date[mask].min():%Y-%m} → {apps.application_date[mask].max():%Y-%m} | vỡ nợ {apps.defaulted[mask].mean():.1%}")
train, val, test, production = (apps[m].copy() for m in splits.values())

Lưu ý thực tế: vì nhãn trễ 12 tháng, nếu triển khai thật ở 01/2024 thì nhãn của hồ sơ 2023 **chưa có** — dữ liệu huấn luyện luôn "cũ" ít nhất một năm. Đó là thêm một lý do phải giám sát bằng nhãn sớm.

## 6. Đặc trưng — và một cái bẫy (chương 05, 17)

In [ ]:
NUMERIC = ["age", "log_income", "loan_amount_vnd", "term_months", "n_dependents", "credit_history_months",
           "late_payments_12m", "dti", "loan_to_income", "income_missing"]
CATEGORICAL = ["employment_type", "purpose", "region", "has_collateral"]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    annuity = CREDIT_MONTHLY_RATE / (1 - (1 + CREDIT_MONTHLY_RATE) ** -out.term_months)
    out["dti"] = out.loan_amount_vnd * annuity / out.monthly_income_vnd                 # khoản trả hằng tháng / thu nhập
    out["log_income"] = np.log(out.monthly_income_vnd)
    out["loan_to_income"] = out.loan_amount_vnd / out.monthly_income_vnd
    out["income_missing"] = out.monthly_income_vnd.isna().astype(int)                   # thiếu thu nhập có thể là tín hiệu
    return out


def make_model(estimator, numeric=NUMERIC):
    columns = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
                                 ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(columns, estimator)


train, val, test, production = (add_features(d) for d in (train, val, test, production))
leaky = make_model(LogisticRegression(max_iter=2000), NUMERIC + ["early_dpd30_3m"]).fit(train, train.defaulted)
honest = make_model(LogisticRegression(max_iter=2000)).fit(train, train.defaulted)
print(f"AUC validation — có early_dpd30_3m: {roc_auc_score(val.defaulted, leaky.predict_proba(val)[:, 1]):.3f} | "
      f"không có: {roc_auc_score(val.defaulted, honest.predict_proba(val)[:, 1]):.3f}")

Thêm `early_dpd30_3m` làm AUC tăng vọt — và **vô dụng**: lúc duyệt hồ sơ, khoản vay chưa tồn tại thì làm sao đã quá hạn? Đây là rò rỉ kinh điển khi bảng dữ liệu chứa cả thông tin trước và sau quyết định. Quy tắc: với mỗi cột, hỏi "giá trị này được ghi **khi nào**?". Từ đây chỉ dùng các đặc trưng có tại thời điểm nộp hồ sơ.

## 7. Baseline và mô hình (chương 07, 12, 14, 19)

So sánh trên **validation** (2023-H2): baseline tỉ lệ nền, luật nghiệp vụ (dùng làm điểm số), hồi quy logistic, gradient boosting mặc định và có điều chuẩn.

In [ ]:
candidates = {
    "hồi quy logistic": make_model(LogisticRegression(max_iter=2000)),
    "HGB mặc định": make_model(HistGradientBoostingClassifier(random_state=0)),
    "HGB điều chuẩn": make_model(HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=8, min_samples_leaf=80,
                                                                l2_regularization=1.0, max_iter=200, random_state=0)),
}
rows = [{"mô hình": "tỉ lệ nền (không mô hình)", "ROC-AUC": 0.5, "PR-AUC": val.defaulted.mean(),
         "Brier": brier_score_loss(val.defaulted, np.full(len(val), train.defaulted.mean()))},
        {"mô hình": "luật: số lần trễ + (DTI > 45%)", "ROC-AUC": roc_auc_score(val.defaulted, val.late_payments_12m + (val.dti > 0.45)),
         "PR-AUC": average_precision_score(val.defaulted, val.late_payments_12m + (val.dti > 0.45)), "Brier": np.nan}]
val_proba = {}
for name, model in candidates.items():
    model.fit(train, train.defaulted)
    val_proba[name] = model.predict_proba(val)[:, 1]
    rows.append({"mô hình": name, "ROC-AUC": roc_auc_score(val.defaulted, val_proba[name]),
                 "PR-AUC": average_precision_score(val.defaulted, val_proba[name]), "Brier": brier_score_loss(val.defaulted, val_proba[name])})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
def paired_bootstrap_auc(y, p_a, p_b, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            diffs.append(roc_auc_score(y[i], p_a[i]) - roc_auc_score(y[i], p_b[i]))
    return np.mean(diffs), np.quantile(diffs, [0.025, 0.975])


mean_diff, ci = paired_bootstrap_auc(val.defaulted, val_proba["hồi quy logistic"], val_proba["HGB điều chuẩn"])
print(f"AUC logistic − HGB điều chuẩn = {mean_diff:+.3f}, KTC 95% bootstrap [{ci[0]:+.3f}, {ci[1]:+.3f}]")

Hồi quy logistic có AUC cao nhất, nhưng khoảng tin cậy của chênh lệch với HGB điều chuẩn **chứa 0** — không thể nói chắc logistic xếp hạng tốt hơn; điều chắc chắn là HGB **không** tốt hơn. Dữ liệu chỉ ~4.500 hồ sơ, ~8% vỡ nợ (≈ 340 ca dương) và quan hệ gần tuyến tính trong log-odds — mô hình phức tạp chỉ thêm phương sai (HGB mặc định overfit rõ). Với hiệu năng tương đương, logistic được chọn vì cho **xác suất hiệu chỉnh tốt** (Brier thấp nhất), **giải thích được** từng quyết định (phần 10 — yêu cầu pháp lý phổ biến trong tín dụng) và chạy nhanh. Chọn hồi quy logistic, tinh chỉnh nhẹ độ điều chuẩn:

In [ ]:
for C in [0.01, 0.1, 1.0, 10.0]:
    m = make_model(LogisticRegression(C=C, max_iter=2000)).fit(train, train.defaulted)
    p = m.predict_proba(val)[:, 1]
    print(f"C = {C:5}: ROC-AUC {roc_auc_score(val.defaulted, p):.4f} | Brier {brier_score_loss(val.defaulted, p):.5f}")

Khác biệt giữa các mức C nhỏ hơn nhiều so với độ bất định (KTC bootstrap rộng ~±0.03) — giữ mặc định `C = 1`. Thử nhiều cấu hình hơn ở đây chỉ làm điểm validation lạc quan thêm (lời nguyền người thắng — chương 39).

## 8. Chính sách quyết định trên validation (chương 14, 19)

In [ ]:
def loan_economics(df):
    gain = MARGIN_PER_YEAR * df.loan_amount_vnd * df.term_months / 12
    loss = LOSS_GIVEN_DEFAULT * df.loan_amount_vnd
    return gain.values, loss.values


def realized_profit(df, approve) -> float:
    gain, loss = loan_economics(df)
    return float(np.where(approve, np.where(df.defaulted.values == 1, -loss, gain), 0).sum() / 1e9)   # tỉ đồng


def policies(df, p_default):
    gain, loss = loan_economics(df)
    return {"duyệt tất cả": np.ones(len(df), bool),
            "luật hiện tại": ((df.late_payments_12m == 0) & (df.dti < 0.45)).values,
            "mô hình (lợi nhuận kỳ vọng > 0)": (1 - p_default) * gain - p_default * loss > 0}


def policy_table(df, p_default):
    rows = []
    for name, approve in policies(df, p_default).items():
        rows.append({"chính sách": name, "tỉ lệ duyệt": approve.mean(), "vỡ nợ trong số được duyệt": df.defaulted.values[approve].mean(),
                     "lợi nhuận (tỉ đồng)": realized_profit(df, approve)})
    rows.append({"chính sách": "(trần lý thuyết: biết trước ai vỡ nợ)", "tỉ lệ duyệt": (df.defaulted == 0).mean(),
                 "vỡ nợ trong số được duyệt": 0.0, "lợi nhuận (tỉ đồng)": realized_profit(df, (df.defaulted == 0).values)})
    return pd.DataFrame(rows)


chosen = make_model(LogisticRegression(max_iter=2000)).fit(train, train.defaulted)
validation_policy = policy_table(val, chosen.predict_proba(val)[:, 1])
print("VALIDATION 2023-H2")
print(validation_policy.round(3).to_string(index=False))

Trên validation, chính sách theo mô hình cho lợi nhuận cao nhất: từ chối ít hồ sơ hơn luật hiện tại nhưng chọn **đúng** những hồ sơ rủi ro. Luật hiện tại quá khắt khe (từ chối cả nhiều khách tốt). Kế hoạch: huấn luyện lại trên train + validation, đánh giá **một lần** trên test.

## 9. Đánh giá cuối trên test 2024-H1 — và một bất ngờ

In [ ]:
development = pd.concat([train, val])
final_model = make_model(LogisticRegression(max_iter=2000)).fit(development, development.defaulted)
p_test = final_model.predict_proba(test)[:, 1]
print(f"TEST 2024-H1: ROC-AUC = {roc_auc_score(test.defaulted, p_test):.3f} | PR-AUC = {average_precision_score(test.defaulted, p_test):.3f} | "
      f"Brier = {brier_score_loss(test.defaulted, p_test):.4f}")
print(f"xác suất dự đoán TB = {p_test.mean():.3f} | tỉ lệ vỡ nợ thật = {test.defaulted.mean():.3f}\n")
print(policy_table(test, p_test).round(3).to_string(index=False))

In [ ]:
reliability = pd.DataFrame({"dự đoán": p_test, "thực tế": test.defaulted.values}).groupby(pd.qcut(p_test, 5, labels=[f"Q{i}" for i in range(1, 6)]), observed=True).mean()
print("calibration theo ngũ phân vị dự đoán (test):")
print(reliability.round(3).T.to_string())

**Bất ngờ**: khả năng **xếp hạng** của mô hình vẫn tốt (AUC không giảm), nhưng **mức** xác suất thấp hơn thực tế 1.5–2 lần ở mọi nhóm — cú sốc kinh tế 2024 làm mọi người rủi ro hơn (trôi khái niệm, chương 40). Vì quyết định dùng ngưỡng trên xác suất, mô hình duyệt quá tay và lợi nhuận thua cả luật cũ (vốn khắt khe "nhờ may" hợp với thời kỳ khó khăn). Một mô hình đánh giá chỉ bằng AUC sẽ không thấy vấn đề này.

## 10. Hậu kiểm: điều gì sẽ cứu được? (chương 37, 40)

**(a) Nhìn lại (dùng nhãn test — chỉ để chẩn đoán, không phải ước lượng hiệu năng):** dịch hệ số chặn (nhân odds với một hằng số) sao cho xác suất trung bình khớp tỉ lệ thật.

In [ ]:
def with_odds_multiplier(p, k):
    return 1 / (1 + np.exp(-(np.log(p / (1 - p)) + np.log(k))))


def odds_multiplier_for_mean(p, target_rate):
    """Hằng số k sao cho trung bình xác suất sau khi nhân odds với k bằng target_rate."""
    return float(np.exp(brentq(lambda log_k: with_odds_multiplier(p, np.exp(log_k)).mean() - target_rate, -5, 5)))


k_hindsight = odds_multiplier_for_mean(p_test, test.defaulted.mean())
approve_recal = policies(test, with_odds_multiplier(p_test, k_hindsight))["mô hình (lợi nhuận kỳ vọng > 0)"]
approve_orig = policies(test, p_test)["mô hình (lợi nhuận kỳ vọng > 0)"]
print(f"nhân odds ×{k_hindsight:.2f}: tỉ lệ duyệt {approve_recal.mean():.1%} | lợi nhuận {realized_profit(test, approve_recal):.2f} tỉ "
      f"(mô hình gốc: {realized_profit(test, approve_orig):.2f}; luật hiện tại: {realized_profit(test, policies(test, p_test)['luật hiện tại']):.2f})")

Với xác suất đúng mức, cùng mô hình (cùng thứ hạng) cho lợi nhuận cao hơn hẳn cả luật cũ — vấn đề nằm ở **calibration**, không ở xếp hạng.

**(b) Làm được ngay lúc đó — bằng nhãn sớm.** Nhãn sớm `early_dpd30_3m` của hồ sơ tháng 01–02/2024 đã có vào **cuối 05/2024**. Nhưng nhãn sớm không phải nhãn thật: nó có độ nhạy $P(\text{sớm}=1\mid\text{vỡ nợ})$ và tỉ lệ dương tính giả $P(\text{sớm}=1\mid\text{không vỡ nợ})$ — ước lượng được trên dữ liệu phát triển (nơi có cả hai nhãn). Từ đó suy ngược tỉ lệ vỡ nợ:

$$\hat\pi_{\text{vỡ nợ}} = \frac{\text{tỉ lệ nhãn sớm} - \text{FPR}}{\text{TPR} - \text{FPR}}$$

rồi hiệu chỉnh hệ số chặn cho hồ sơ từ **06/2024** — không dùng nhãn 12 tháng nào của 2024.

In [ ]:
tpr = development.early_dpd30_3m[development.defaulted == 1].mean()
fpr = development.early_dpd30_3m[development.defaulted == 0].mean()
known_by_june = test[test.application_date < "2024-03-01"]
estimated_rate = (known_by_june.early_dpd30_3m.mean() - fpr) / (tpr - fpr)
k_proxy = odds_multiplier_for_mean(final_model.predict_proba(known_by_june)[:, 1], estimated_rate)
print(f"nhãn sớm trên dữ liệu phát triển: TPR = {tpr:.3f}, FPR = {fpr:.3f}")
print(f"hồ sơ 01–02/2024: tỉ lệ nhãn sớm {known_by_june.early_dpd30_3m.mean():.3f} → ước lượng vỡ nợ {estimated_rate:.3f} "
      f"(thật, biết sau 1 năm: {known_by_june.defaulted.mean():.3f}) | mô hình dự đoán {final_model.predict_proba(known_by_june)[:, 1].mean():.3f} → nhân odds ×{k_proxy:.2f}")

june = test[test.application_date >= "2024-06-01"]
p_june = final_model.predict_proba(june)[:, 1]
compare = {"luật hiện tại": policies(june, p_june)["luật hiện tại"],
           "mô hình gốc": policies(june, p_june)["mô hình (lợi nhuận kỳ vọng > 0)"],
           f"mô hình hiệu chỉnh theo nhãn sớm (×{k_proxy:.2f})": policies(june, with_odds_multiplier(p_june, k_proxy))["mô hình (lợi nhuận kỳ vọng > 0)"]}
print(f"\nhồ sơ 06/2024 ({len(june)} hồ sơ, vỡ nợ thật {june.defaulted.mean():.1%}):")
for name, ap in compare.items():
    print(f"   {name:45s}: duyệt {ap.mean():.1%} | vỡ nợ trong số duyệt {june.defaulted.values[ap].mean():.1%} | lợi nhuận {realized_profit(june, ap):5.2f} tỉ")

Sau 5 tháng (thay vì hơn một năm), mức rủi ro ước lượng từ nhãn sớm đã gần đúng, và chính sách hiệu chỉnh vượt cả luật cũ lẫn mô hình gốc trên hồ sơ tháng 6. (Chỉ một tháng ~340 hồ sơ — con số lợi nhuận còn nhiễu; điều đáng tin là ước lượng **mức rủi ro**.) Giả định của cách này: quan hệ giữa nhãn sớm và vỡ nợ (TPR, FPR) không đổi khi có cú sốc, và cú sốc nhân odds đều cho mọi nhóm — cần theo dõi khi nhãn thật về. Bài học cho hệ thống: (1) giám sát **calibration** chứ không chỉ AUC; (2) có **nhãn sớm** và quy trình **hiệu chỉnh hệ số chặn** nhanh (rẻ, ít rủi ro) tách khỏi huấn luyện lại đầy đủ; (3) **stress test** chính sách với kịch bản rủi ro tăng ×1.5–2 trước khi triển khai.

## 11. Giải thích (chương 36)

Trong tín dụng, người bị từ chối thường có quyền biết **lý do** (adverse action reasons). Với hồi quy logistic, đóng góp của mỗi đặc trưng vào log-odds là `hệ số × giá trị đã chuẩn hoá` — chính xác, không cần xấp xỉ.

In [ ]:
pre, clf = final_model[0], final_model[-1]
names = pre.get_feature_names_out()
coefs = pd.Series(clf.coef_[0], index=names)
print("hệ số lớn nhất (thang chuẩn hoá — +: tăng rủi ro):")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

In [ ]:
REASON_TEXT = {"num__dti": "tỉ lệ trả nợ/thu nhập cao", "num__late_payments_12m": "có trễ hạn trong 12 tháng",
               "num__log_income": "thu nhập thấp", "num__credit_history_months": "lịch sử tín dụng ngắn",
               "cat__employment_type_unemployed": "không có việc làm", "cat__employment_type_contract": "lao động hợp đồng",
               "cat__has_collateral_False": "không có tài sản đảm bảo", "num__loan_to_income": "khoản vay lớn so với thu nhập",
               "num__n_dependents": "nhiều người phụ thuộc", "cat__purpose_business": "mục đích kinh doanh"}


def reason_codes(model, applicant: pd.DataFrame, top=3):
    x = model[0].transform(applicant)
    x = x.toarray() if hasattr(x, "toarray") else x
    contrib = pd.Series(x[0] * model[-1].coef_[0], index=model[0].get_feature_names_out())
    return [(REASON_TEXT.get(name, name), round(value, 2)) for name, value in contrib.sort_values(ascending=False).head(top).items() if value > 0]


gain, loss = loan_economics(test)
rejected = test[(1 - p_test) * gain - p_test * loss <= 0]
example = rejected.sort_values("application_date").iloc[[0]]
print(example[["application_id", "age", "employment_type", "monthly_income_vnd", "loan_amount_vnd", "term_months", "late_payments_12m", "has_collateral"]].to_string(index=False))
g_ex, l_ex = loan_economics(example)
print(f"P(vỡ nợ) = {final_model.predict_proba(example)[0, 1]:.1%} > ngưỡng hoà vốn {g_ex[0] / (g_ex[0] + l_ex[0]):.1%} → từ chối. Lý do chính:")
for text, value in reason_codes(final_model, example):
    print(f"   • {text} (+{value} log-odds)")

## 12. Kiểm tra theo nhóm (chương 36)

In [ ]:
def group_report(df, p, group):
    gain, loss = loan_economics(df)
    approve = (1 - p) * gain - p * loss > 0
    rows = []
    for g, idx in df.groupby(group, observed=True).indices.items():
        y = df.defaulted.values[idx]
        rows.append({group: g, "hồ sơ": len(idx), "vỡ nợ thật": y.mean(), "dự đoán TB": p[idx].mean(),
                     "ROC-AUC": roc_auc_score(y, p[idx]) if 0 < y.mean() < 1 else np.nan,
                     "tỉ lệ duyệt": approve[idx].mean(), "vỡ nợ trong số duyệt": y[approve[idx]].mean()})
    return pd.DataFrame(rows)


test["age_group"] = pd.cut(test.age, [17, 30, 45, 100], labels=["< 30", "30–45", "> 45"])
print("theo vùng (validation, mô hình chỉ học trên train):")
print(group_report(val, chosen.predict_proba(val)[:, 1], "region").round(3).to_string(index=False))
print("\ntheo nhóm tuổi (test):")
print(group_report(test, p_test, "age_group").round(3).to_string(index=False))

Theo vùng: hiệu năng, calibration và tỉ lệ duyệt tương đương. Theo tuổi (test, trong cú sốc): tỉ lệ duyệt chênh ít, nhưng mô hình đánh giá thấp rủi ro của người **trẻ** nhiều hơn (dự đoán chỉ bằng khoảng một nửa tỉ lệ thật) — nên trong số hồ sơ được duyệt, tỉ lệ vỡ nợ của nhóm < 30 tuổi cao gần gấp đôi nhóm > 45. Tuổi không phải đặc trưng của mô hình, nhưng `credit_history_months` và loại việc làm (nhóm chịu cú sốc mạnh) tương quan với tuổi. **Calibration theo nhóm** vì vậy là một guardrail cần giám sát; còn đặc trưng nào được phép dùng, chênh lệch nào chấp nhận được là câu hỏi **chính sách** cần đưa ra cùng bộ phận pháp chế: đặc trưng nào được phép dùng, chênh lệch nào chấp nhận được, có cần sản phẩm riêng cho khách trẻ không (chương 36: công bằng không có một định nghĩa duy nhất).

## 13. Đóng gói (chương 39, 40)

Gói gồm: hàm làm sạch + đặc trưng + mô hình (một hàm `score` duy nhất cho phục vụ), metadata, model card.

In [ ]:
def score_application(record: dict) -> dict:
    """Hàm phục vụ: bản ghi thô (như từ form) → làm sạch → đặc trưng → xác suất → quyết định + lý do."""
    df = add_features(clean_applications(pd.DataFrame([record])))
    problems = validate_clean(df)
    if problems:
        return {"status": "rejected_input", "problems": problems}
    p = float(final_model.predict_proba(df)[0, 1])
    gain, loss = loan_economics(df)
    decision = "duyệt" if (1 - p) * gain[0] - p * loss[0] > 0 else "từ chối"
    return {"status": "ok", "p_default": round(p, 4), "decision": decision,
            "reasons": [t for t, _ in reason_codes(final_model, df)] if decision == "từ chối" else []}


form = raw.drop(columns=["defaulted", "early_dpd30_3m"]).iloc[0].to_dict()   # một hồ sơ thô bất kỳ (kể cả định dạng lộn xộn)
print(score_application(form))
print(score_application({**form, "region": "Sao Hoả"}))

In [ ]:
package_dir = ARTIFACTS / "credit_model_v1"
package_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "name": "credit_default_12m", "version": "1.0.0", "trained_on": "hồ sơ 2022-01 → 2023-12 (nhãn 12 tháng)",
    "features": NUMERIC + CATEGORICAL, "excluded": ["early_dpd30_3m (sau quyết định)", "application_date", "id"],
    "economics": {"margin_per_year": MARGIN_PER_YEAR, "loss_given_default": LOSS_GIVEN_DEFAULT},
    "validation_2023H2": {"roc_auc": round(roc_auc_score(val.defaulted, val_proba["hồi quy logistic"]), 4)},
    "test_2024H1": {"roc_auc": round(roc_auc_score(test.defaulted, p_test), 4), "mean_pred": round(p_test.mean(), 4), "default_rate": round(test.defaulted.mean(), 4)},
    "known_issues": ["đánh giá thấp rủi ro khi có cú sốc kinh tế — cần hiệu chỉnh hệ số chặn theo nhãn sớm"],
    "sklearn": sklearn.__version__,
}
(package_dir / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metadata["test_2024H1"]), "→", package_dir)

Tiếp theo (đã học ở chương 39–40, không lặp lại ở đây): lưu `final_model` bằng joblib/skops kèm hash, viết API FastAPI gọi `score_application`, kiểm thử parity, Dockerfile, CI.

## 14. Kế hoạch giám sát (chương 40)

Hai câu hỏi: (1) giám sát **đầu vào** có thấy cú sốc không? (2) **nhãn sớm** thấy sau bao lâu?

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    edges = np.unique(np.quantile(reference.dropna(), np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.clip(np.histogram(reference.dropna(), edges)[0] / reference.notna().sum(), eps, None)
    c = np.clip(np.histogram(current.dropna(), edges)[0] / current.notna().sum(), eps, None)
    return float(np.sum((c - r) * np.log(c / r)))


later_periods = {"test 2024-H1": test, "production 2024-H2→": production}
drift = pd.DataFrame({name: {col: psi(development[col], d[col]) for col in ["dti", "log_income", "loan_amount_vnd", "credit_history_months", "late_payments_12m"]}
                      | {"xác suất dự đoán": psi(pd.Series(final_model.predict_proba(development)[:, 1]), pd.Series(final_model.predict_proba(d)[:, 1]))}
                      for name, d in later_periods.items()})
print("PSI so với dữ liệu phát triển:")
print(drift.round(3).to_string())

In [ ]:
monthly = apps.assign(month=apps.application_date.dt.to_period("M")).groupby("month").agg(early=("early_dpd30_3m", "mean"), default=("defaulted", "mean"))
monthly.index = monthly.index.to_timestamp()
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(monthly.index, monthly.early, marker=".", label="nhãn sớm: quá hạn 30 ngày trong 3 tháng (có sau 3 tháng)")
ax.plot(monthly.index, monthly.default, marker=".", label="vỡ nợ 12 tháng (có sau 12 tháng)")
ax.axvline(pd.Timestamp("2024-01-01"), color="red", ls="--", lw=1, label="cú sốc 01/2024")
ax.axhline(development.early_dpd30_3m.mean() * 1.3, color="gray", ls=":", lw=1, label="ngưỡng cảnh báo nhãn sớm (+30%)")
ax.set_ylabel("tỉ lệ theo tháng nộp hồ sơ")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

- **Đầu vào ổn định** (PSI nhỏ): hồ sơ nộp vào trông giống hệt trước — giám sát dữ liệu đầu vào và phân phối dự đoán **không** báo động. Cú sốc là trôi khái niệm.
- **Nhãn sớm** tăng rõ ngay từ các tháng đầu 2024, quan sát được sau ~3 tháng; nhãn 12 tháng chỉ xác nhận sau một năm.

Kế hoạch giám sát đưa vào design doc:

| Tín hiệu | Tần suất | Ngưỡng | Hành động |
|---|---|---|---|
| PSI đặc trưng chính, phân phối dự đoán | hằng tuần | PSI > 0.25 | điều tra nguồn dữ liệu |
| Nhãn sớm (dpd30 trong 3 tháng) theo tháng nộp | hằng tháng | > 1.3 × mức phát triển trong 2 tháng liên tiếp | hiệu chỉnh hệ số chặn; báo cáo rủi ro |
| Calibration với nhãn 12 tháng | hằng quý | \|TB dự đoán − thực tế\| > 2 điểm % | huấn luyện lại + cổng thăng hạng |
| Tỉ lệ duyệt theo vùng/tuổi | hằng tháng | lệch > 10 điểm % so với kỳ trước | rà soát chính sách |
| Độ trễ, lỗi API | liên tục | p99 > 50 ms, lỗi > 0.5% | trực vận hành |

## 15. Tóm tắt kiểu báo cáo capstone

In [ ]:
val_tbl = validation_policy.set_index("chính sách")["lợi nhuận (tỉ đồng)"]
test_tbl = policy_table(test, p_test).set_index("chính sách")["lợi nhuận (tỉ đồng)"]
summary = f"""TÓM TẮT
• Dữ liệu: {len(raw):,} dòng thô → {len(apps):,} hồ sơ sạch (đạt hợp đồng dữ liệu); nhãn vỡ nợ 12 tháng, có đến 06/2024.
• Mô hình: hồi quy logistic trên 14 đặc trưng tại thời điểm nộp hồ sơ (loại cột nhãn sớm vì rò rỉ);
  ngang gradient boosting về xếp hạng (KTC chênh lệch chứa 0) nhưng hiệu chỉnh tốt hơn và giải thích được.
• Validation 2023-H2: chính sách theo mô hình lãi {val_tbl.iloc[2]:.1f} tỉ vs luật hiện tại {val_tbl.iloc[1]:.1f} tỉ.
• Test 2024-H1 (cú sốc kinh tế): AUC {roc_auc_score(test.defaulted, p_test):.3f} vẫn tốt nhưng xác suất thấp hơn thực tế (TB {p_test.mean():.1%} vs {test.defaulted.mean():.1%});
  chính sách theo mô hình lãi {test_tbl.iloc[2]:.1f} tỉ < luật hiện tại {test_tbl.iloc[1]:.1f} tỉ. Nhãn sớm ước lượng lại mức rủi ro sau 5 tháng\n  (vỡ nợ ước lượng {estimated_rate:.1%}, thật {known_by_june.defaulted.mean():.1%}) → hiệu chỉnh hệ số chặn ×{k_proxy:.2f}.
• Khuyến nghị: triển khai mô hình KÈM giám sát nhãn sớm và quy trình hiệu chỉnh hệ số chặn; shadow 1 tháng, rồi A/B (chương 40);
  theo dõi calibration theo nhóm tuổi (người trẻ bị đánh giá thấp rủi ro nhiều nhất) và rà soát với pháp chế.
• Hạn chế: dữ liệu mô phỏng; chưa có thông tin tín dụng ngoài (CIC); điều chỉnh odds đều giả định cú sốc tác động như nhau."""
print(summary)

## 🧠 Kiểm tra nhanh

1. Kể ba loại vấn đề trong dữ liệu thô và cách xử lý mỗi loại.
2. Vì sao `early_dpd30_3m` bị loại khỏi đặc trưng nhưng lại hữu ích cho giám sát?
3. Vì sao chia theo thời gian, không chia ngẫu nhiên?
4. Ngưỡng duyệt được suy ra thế nào từ kinh tế khoản vay? Vì sao calibration quan trọng hơn AUC ở đây?
5. Mô hình có AUC test tốt nhưng lỗ hơn luật cũ — giải thích. Cách nào phát hiện và khắc phục sớm?
6. Lý do từ chối (reason codes) tính thế nào với hồi quy logistic?
7. Vì sao giám sát đầu vào không thấy cú sốc 2024?

## 📌 Tóm tắt

- Dữ liệu thô cần **kiểm tra** có hệ thống, **làm sạch** bằng code có hợp đồng dữ liệu, và hiểu **ngữ nghĩa thời gian** của từng cột.
- Chia theo thời gian; baseline nghiệp vụ; mô hình đơn giản, hiệu chỉnh tốt, giải thích được thường là lựa chọn đúng với dữ liệu bảng vừa và nhỏ.
- Quyết định theo **lợi nhuận kỳ vọng** cần **xác suất đúng mức**, không chỉ xếp hạng đúng; stress test trước khi triển khai.
- Thế giới thay đổi: giám sát bằng **nhãn sớm**, hiệu chỉnh nhanh, báo cáo trung thực cả điều không hiệu quả.

## 📚 Tài liệu tham khảo

- Siddiqi (2017) — *Intelligent Credit Scoring* (2nd ed.)
- Thomas, Crook & Edelman (2017) — *Credit Scoring and Its Applications*
- Basel Committee — nguyên tắc quản lý rủi ro mô hình; SR 11-7 (Fed) — *Guidance on Model Risk Management*
- Huyen (2022) — *Designing Machine Learning Systems*
- Các chương 03–05, 07, 14–17, 19, 36–40 của khoá học